Подключение необходимых библиотек и загрузка данных для обучения и тестирования

In [ ]:
import pandas 
import numpy
import random
import math
import seaborn

# Data Visualization
import matplotlib.pyplot as plt

%matplotlib inline

# ML
import sklearn
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_squared_log_error

train_data = pandas.read_csv("../input/house-prices-advanced-regression-techniques/train.csv")
test_data = pandas.read_csv("../input/house-prices-advanced-regression-techniques/test.csv")

Проверим выборки на наличие выбросов. Для этого выведем тепловую карту попарных сравнений и таблицу корреляции с целевым параметром SalePrice

In [ ]:
corrmat = train_data.corr()
k = 80 # количество коррелирующих признаков, которое мы хотим увидеть
fig, ax = plt.subplots(figsize = (14,14))
cols = corrmat.nlargest(k, 'SalePrice')['SalePrice'].index
cm = numpy.corrcoef(train_data[cols].values.T)
seaborn.set(font_scale=1.25)
hm = seaborn.heatmap(cm, cbar=True, annot=True, square=True, 
                 fmt='.2f', annot_kws={'size': 10}, 
                 yticklabels=cols.values, xticklabels=cols.values)
plt.show()

corrmat = train_data.corr()
k = 13 # количество коррелирующих признаков, которое мы хотим увидеть
corrmat.nlargest(k, 'SalePrice')['SalePrice']

Выведем графики разброса для самых коррелирующих с SalePrices признаков : GrLivArea, OverallQual, GarageArea, TotalBsmtSF, MasVnrArea

In [ ]:

#Проверим наличие выбросов в GrLivArea
fig, ax = plt.subplots()
ax.scatter(x = train_data['GrLivArea'], y = train_data['SalePrice'])
plt.ylabel('SalePrice', fontsize=13)
plt.xlabel('GrLivArea', fontsize=13)
plt.show()
#Проверим наличие выбросов в OverallQual
fig, ax = plt.subplots()
ax.scatter(x = train_data['OverallQual'], y = train_data['SalePrice'])
plt.ylabel('SalePrice', fontsize=13)
plt.xlabel('OverallQual', fontsize=13)
plt.show()
#Проверим наличие выбросов в GarageArea
fig, ax = plt.subplots()
ax.scatter(x = train_data['GarageArea'], y = train_data['SalePrice'])
plt.ylabel('SalePrice', fontsize=13)
plt.xlabel('GarageArea', fontsize=13)
plt.show()
#Проверим наличие выбросов в TotalBsmtSF
fig, ax = plt.subplots()
ax.scatter(x = train_data['TotalBsmtSF'], y = train_data['SalePrice'])
plt.ylabel('SalePrice', fontsize=13)
plt.xlabel('TotalBsmtSF', fontsize=13)
plt.show()
#Проверим наличие выбросов в MasVnrArea
fig, ax = plt.subplots()
ax.scatter(x = train_data['MasVnrArea'], y = train_data['SalePrice'])
plt.ylabel('SalePrice', fontsize=13)
plt.xlabel('MasVnrArea', fontsize=13)
plt.show()

Ликвидируем выбросы

In [ ]:
train_data = train_data.drop(train_data[(train_data['OverallQual'] > 9) & (train_data['SalePrice'] < 220000)].index)
train_data = train_data.drop(train_data[(train_data['GrLivArea'] > 4000) & (train_data['SalePrice'] < 300000)].index)
train_data = train_data.drop(train_data[(train_data['TotalBsmtSF'] > 6000)].index)
train_data = train_data.drop(train_data[(train_data['GarageArea'] > 1200) & (train_data['SalePrice'] < 300000)].index)
train_data = train_data.drop(train_data[(train_data['SalePrice'] > 650000)].index)
train_data = train_data.drop(train_data[(train_data['MasVnrArea'] > 1200)].index)

Проверим данные на наличие неинициализированных объектов

In [ ]:
train_data.isnull().sum().sort_values(ascending=False).head(20)

Далее мы будем проводить операции которые нужно совершать и с тренировочными и с тестовыми данными, поэтому разумно будет обьеденить данные.

In [ ]:
Target = 'SalePrice'
train_data.dropna(axis=0, subset=[Target], inplace=True)

all_data = pandas.concat([train_data, test_data],keys=['train','test'])

all_data = all_data.drop(columns=['Id'], axis=1)

Обработаем пропущенные значения. Пропущенные категориальные признаки заменяются заглушкой "UNKNOWN", а числовые на рандомизированное значение в интервале от среднего минус среднеквадратичное отклонение, до среднего плюс среднеквадратичное отклонение

In [ ]:
def ResolveMissingValues(df):
    num_cols = [cname for cname in df.columns if df[cname].dtype in ['int64', 'float64']]
    dog_cols = [cname for cname in df.columns if df[cname].dtype == "object"]
    values = {}
    for a in dog_cols:
        values[a] = 'UNKNOWN'
    for a in num_cols:
        mean1=df[a].mean() 
        std1=df[a].std()
        values[a]=random.randint(int(mean1-std1), int(mean1+std1))
        
    df.fillna(value=values, inplace=True)
    
ResolveMissingValues(all_data)

После этого у нас нет пропущенных значений

In [ ]:
all_data.isnull().sum().sum()

Далее необходимо заменить категориальные значения на числовые. Столбец с категориальным признаком заменим на n бинарных столбцов, где n это количетсво уникальных значений изначального столбца

In [ ]:
def getObjectColumnsList(df):
    return [cname for cname in df.columns if df[cname].dtype == "object"]

def PerformOneHotEncoding(df, columnsToEncode):
    return pandas.get_dummies(df, columns=columnsToEncode)

dog_cols = getObjectColumnsList(all_data)
all_data = PerformOneHotEncoding(all_data, dog_cols)
all_data.head()

Разьеденим ранее объединённые тестовые и тренировочные данные

In [ ]:
train_data = all_data.loc['train']
test_data = all_data.loc['test']
train_data.shape,test_data.shape

target = train_data['SalePrice']
train_data = train_data.drop(['SalePrice'], axis=1)
test_data = test_data.drop(['SalePrice'], axis=1)

X, y = train_data, target

Среди различных методов был выбран GradientBoostingRegressor, так как показал наибольшую эффективность. Параметры функции так же были подобраны перебором. 

In [ ]:
gbr_reg = GradientBoostingRegressor(n_estimators=2000, learning_rate=0.05,max_depth=3, max_features='sqrt',min_samples_leaf=15, min_samples_split=10,loss='huber')
gbr_reg.fit(X, y)
pred = gbr_reg.predict(X)
print(gbr_reg.score(X, y))
numpy.sqrt(mean_squared_log_error(pred, y))

Проверка на тестовых данных и запись в файл

In [ ]:
myPrediction = gbr_reg.predict(test_data)
submission = pandas.DataFrame({
        "Id": list(range(1461, 1461+len(test_data))),
        "SalePrice": myPrediction
    })
submission.to_csv('submission.csv', index=False)